<a href="https://colab.research.google.com/github/Abbta/Abbta.github.io/blob/master/AI_security_HW1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# ----------------------------
# Config
# ----------------------------
BATCH_SIZE_TRAIN = 64
BATCH_SIZE_TEST = 1000
EPOCHS = 3
LR = 0.001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPSILONS = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25]
MODEL_PATH = "mnist_cnn.pt"
RESULTS_DIR = "results"

os.makedirs(RESULTS_DIR, exist_ok=True)

# ----------------------------
# Data
# ----------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_loader = torch.utils.data.DataLoader(
    datasets.MNIST("./data", train=True, download=True, transform=transform),
    batch_size=BATCH_SIZE_TRAIN,
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    datasets.MNIST("./data", train=False, download=True, transform=transform),
    batch_size=1,
    shuffle=True
)

# ----------------------------
# Model
# ----------------------------
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(64 * 12 * 12, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))      # 28 -> 26
        x = F.relu(self.conv2(x))      # 26 -> 24
        x = self.pool(x)               # 24 -> 12
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SmallCNN().to(DEVICE)

# ----------------------------
# Train
# ----------------------------
def train():
    optimizer = optim.Adam(model.parameters(), lr=LR)
    model.train()

    for epoch in range(EPOCHS):
        running_loss = 0.0
        correct = 0
        total = 0

        for data, target in train_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)

            optimizer.zero_grad()
            output = model(data)
            loss = F.cross_entropy(output, target)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * data.size(0)
            pred = output.argmax(dim=1)
            correct += (pred == target).sum().item()
            total += target.size(0)

        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {running_loss/total:.4f} | Train Acc: {correct/total:.4f}")

    torch.save(model.state_dict(), MODEL_PATH)
    print(f"Saved model to {MODEL_PATH}")

# ----------------------------
# Clean test accuracy
# ----------------------------
def test_clean():
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            output = model(data)
            pred = output.argmax(dim=1)
            correct += (pred == target).sum().item()
            total += 1

    acc = correct / total
    print(f"Clean test accuracy: {acc:.4f}")
    return acc

# ----------------------------
# FGSM attack
# ----------------------------
def fgsm_attack(image, epsilon, data_grad):
    sign_data_grad = data_grad.sign()
    perturbed_image = image + epsilon * sign_data_grad
    perturbed_image = torch.clamp(perturbed_image, 0, 1)
    return perturbed_image

# ----------------------------
# Evaluate attack
# ----------------------------
def evaluate_fgsm(epsilon, max_examples=5):
    correct = 0
    adv_examples = []

    model.eval()

    for data, target in test_loader:
        data, target = data.to(DEVICE), target.to(DEVICE)
        data.requires_grad = True

        output = model(data)
        init_pred = output.argmax(dim=1)

        # Only attack samples that the model already got correct
        if init_pred.item() != target.item():
            continue

        loss = F.cross_entropy(output, target)
        model.zero_grad()
        loss.backward()

        data_grad = data.grad.data
        perturbed_data = fgsm_attack(data, epsilon, data_grad)

        output_adv = model(perturbed_data)
        final_pred = output_adv.argmax(dim=1)

        if final_pred.item() == target.item():
            correct += 1
        else:
            if len(adv_examples) < max_examples:
                adv_examples.append((
                    target.item(),
                    init_pred.item(),
                    final_pred.item(),
                    data.squeeze().detach().cpu(),
                    perturbed_data.squeeze().detach().cpu()
                ))

    # Count only originally correct samples
    total_considered = 0
    model.eval()
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            output = model(data)
            pred = output.argmax(dim=1)
            if pred.item() == target.item():
                total_considered += 1

    acc = correct / total_considered
    print(f"Epsilon: {epsilon:.2f} | Accuracy under attack: {acc:.4f}")
    return acc, adv_examples

# ----------------------------
# Plot examples
# ----------------------------
def save_adv_examples(all_examples):
    for epsilon, examples in all_examples.items():
        if len(examples) == 0:
            continue

        n = len(examples)
        fig, axes = plt.subplots(n, 2, figsize=(4, 2*n))
        if n == 1:
            axes = [axes]

        for i, ex in enumerate(examples):
            true_label, init_pred, final_pred, original, attacked = ex

            axes[i][0].imshow(original, cmap="gray")
            axes[i][0].set_title(f"Original\ntrue={true_label}, pred={init_pred}")
            axes[i][0].axis("off")

            axes[i][1].imshow(attacked, cmap="gray")
            axes[i][1].set_title(f"Attacked\ntrue={true_label}, pred={final_pred}")
            axes[i][1].axis("off")

        plt.tight_layout()
        plt.savefig(os.path.join(RESULTS_DIR, f"adv_examples_eps_{str(epsilon).replace('.', '_')}.png"))
        plt.close()

# ----------------------------
# Plot accuracy curve
# ----------------------------
def save_accuracy_plot(epsilons, accuracies):
    plt.figure(figsize=(6, 4))
    plt.plot(epsilons, accuracies, marker="o")
    plt.xlabel("Epsilon")
    plt.ylabel("Accuracy under FGSM attack")
    plt.title("FGSM attack on MNIST classifier")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "fgsm_accuracy_curve.png"))
    plt.close()

# ----------------------------
# Main
# ----------------------------
if __name__ == "__main__":
    if not os.path.exists(MODEL_PATH):
        train()
    else:
        model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
        print(f"Loaded model from {MODEL_PATH}")

    clean_acc = test_clean()

    accuracies = []
    all_examples = {}

    for eps in EPSILONS:
        acc, examples = evaluate_fgsm(eps)
        accuracies.append(acc)
        all_examples[eps] = examples

    save_accuracy_plot(EPSILONS, accuracies)
    save_adv_examples(all_examples)

    print("\nDone.")
    print(f"Clean accuracy: {clean_acc:.4f}")
    print("Attack accuracies:")
    for eps, acc in zip(EPSILONS, accuracies):
        print(f"  eps={eps:.2f}: {acc:.4f}")

100%|██████████| 9.91M/9.91M [00:00<00:00, 45.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.16MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.7MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.79MB/s]


Epoch 1/3 | Loss: 0.1534 | Train Acc: 0.9530
Epoch 2/3 | Loss: 0.0416 | Train Acc: 0.9874
Epoch 3/3 | Loss: 0.0260 | Train Acc: 0.9918
Saved model to mnist_cnn.pt
Clean test accuracy: 0.9888
Epsilon: 0.00 | Accuracy under attack: 1.0000
Epsilon: 0.05 | Accuracy under attack: 0.9700
Epsilon: 0.10 | Accuracy under attack: 0.8858
Epsilon: 0.15 | Accuracy under attack: 0.6993
Epsilon: 0.20 | Accuracy under attack: 0.3845
Epsilon: 0.25 | Accuracy under attack: 0.1609

Done.
Clean accuracy: 0.9888
Attack accuracies:
  eps=0.00: 1.0000
  eps=0.05: 0.9700
  eps=0.10: 0.8858
  eps=0.15: 0.6993
  eps=0.20: 0.3845
  eps=0.25: 0.1609
